# 🔒 Privacy Filter — Finetuning Notebook
**openai/privacy-filter** — Fine-tuning supervisionato su documenti italiani e dominio legale, con `opf train`.

Il dataset sintetico (3400 esempi: documenti italiani + dominio legale) è generato automaticamente dal modulo `dataset_builder.py`. Tutti i dati sono **sintetici** — nomi, codici fiscali, IBAN generati casualmente ma nel formato corretto, senza dati personali reali.

---
## Indice

**Setup**
1. Installazione dipendenze
2. Verifica device (CUDA / MPS / CPU)

**Generazione dataset sintetico**
3. Generatori di formati italiani
4. Dataset Step 1 — Documenti d'identità italiani
5. Dataset Step 2 — Dominio legale italiano
6. Validazione e salvataggio `.jsonl` (6 file: train/val/test × step1/step2)

**Training**
7. Training Step 1 — Documenti italiani (parte dal modello base `openai/privacy-filter`)
8. Training Step 2 — Dominio legale (parte dal checkpoint Step 1)

**Valutazione**
9. Test qualitativo su frasi di esempio
10. Valutazione quantitativa su test set held-out (precision/recall/F1)

**Salvataggio**
11. Salvataggio checkpoint su Google Drive (Colab) o cartella locale (Mac)


## 1. Installazione dipendenze
Installa il package `opf` dal repo GitHub ufficiale (non è pubblicato su PyPI) e le librerie necessarie.
> ⚠️ Il kernel si riavvierà automaticamente dopo l'installazione.

In [1]:
# Cella 1 — Installazione
# Il package si chiama 'opf' e non è pubblicato su PyPI: installazione diretta dal repo GitHub
!pip install -q git+https://github.com/openai/privacy-filter.git

# Dipendenze aggiuntive per il finetuning
!pip install -q accelerate safetensors tiktoken

# Verifica CLI
!opf --help

print('\n✅ Installazione completata!')

usage: opf [-h]

OpenAI Privacy Filter (OPF): redact text to remove PII. Redact locally via CLI and interactive mode; run evaluations; or fine-tune on your own labeled dataset.

Subcommands:
  redact  Redact text locally (default, implied).
  eval    Run encoder eval on a ground-truth dataset.
  train   Fine-tune a checkpoint on a local labeled dataset.
Default mode: redact
  The redact mode has additional flags; see `opf redact --help`.

options:
  -h, --help  show this help message and exit

✅ Installazione completata!


## 2. Verifica GPU
Controlla se è disponibile una GPU. Il finetuning è molto più veloce con GPU.

In [2]:
# Cella 2 — Verifica device (CUDA / MPS Apple Silicon / CPU) e ambiente
import torch, sys, platform

# ── Rilevamento ambiente ──
IS_COLAB = 'google.colab' in sys.modules
IS_MAC   = platform.system() == 'Darwin'

# ── Rilevamento device ──
if torch.cuda.is_available():
    DEVICE_CLI = 'cuda'          # passato a `opf train --device`
    DEVICE_HF  = 0               # passato a transformers.pipeline(device=...)
    print(f'✅ CUDA: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE_CLI = 'mps'           # Apple Silicon (M1/M2/M3/M4)
    DEVICE_HF  = 'mps'
    print(f'✅ MPS (Apple Silicon): {platform.processor() or platform.machine()}')
    print(f'   Memoria unificata: {round(torch.mps.driver_allocated_memory()/1e9, 2) if hasattr(torch.mps, "driver_allocated_memory") else "—"} GB allocati')
else:
    DEVICE_CLI = 'cpu'
    DEVICE_HF  = -1
    print('⚠️  Nessuna GPU/MPS rilevata — training su CPU (molto più lento)')

print(f'\nPyTorch: {torch.__version__}')
print(f'Ambiente: {"Colab" if IS_COLAB else ("macOS locale" if IS_MAC else "Linux/Windows locale")}')
print(f'Device CLI: --device {DEVICE_CLI}   |   Device HF pipeline: {DEVICE_HF}')


✅ MPS (Apple Silicon): arm
   Memoria unificata: 0.0 GB allocati

PyTorch: 2.11.0
Ambiente: macOS locale
Device CLI: --device mps   |   Device HF pipeline: mps


## 3. Generatori di formati italiani

Funzioni che producono valori sintetici ma strutturalmente validi per ogni tipo di documento.

In [3]:
# S2 — Generatori importati da dataset_builder.py
# Tutti i generatori (nomi, CF, IBAN, patente, ecc.) sono definiti nel modulo.
# Per personalizzare liste di nomi/comuni, edita le costanti in dataset_builder.py

from dataset_builder import (
    rand_nome, gen_cf, gen_ci, gen_patente, gen_passaporto,
    gen_piva, gen_iban, gen_ts, gen_telefono, gen_email,
    gen_procedimento, gen_catastale,
)

# Smoke-test: genera un esempio di ogni tipo
cf, nome, cognome = gen_cf()
print('Esempi generati dal modulo:')
print(f'  Nome:             {nome} {cognome}')
print(f'  Codice Fiscale:   {cf}')
print(f'  Carta Identità:   {gen_ci()}')
print(f'  Patente:          {gen_patente()}')
print(f'  Passaporto:       {gen_passaporto()}')
print(f'  Partita IVA:      {gen_piva()}')
print(f'  IBAN:             {gen_iban()}')
print(f'  Tessera Sanit.:   {gen_ts(cf)}')
print(f'  Telefono:         {gen_telefono()}')
print(f'  Email:            {gen_email(nome, cognome)}')
print(f'  Procedimento:     {gen_procedimento()}')
print(f'  Catastale:        {gen_catastale()}')
print('\n✅ Generatori pronti!')


Esempi generati dal modulo:
  Nome:             Lorenzo Ferrari
  Codice Fiscale:   FRRLNZ66P10G273V
  Carta Identità:   WH8722363
  Patente:          YE064WV67664E
  Passaporto:       AR6708544
  Partita IVA:      96399191913
  IBAN:             IT09EN91403481390XPN44SC07T
  Tessera Sanit.:   800388672325FRRLNZ66P10G273V
  Telefono:         +39 349 170 6610
  Email:            ferrari.lorenzo@yahoo.it
  Procedimento:     n. 5932/2024 RG
  Catastale:        foglio 397, mappale 3930, sub. 15, sez. C

✅ Generatori pronti!


## 4. Dataset Step 1 — Documenti d'identità italiani

Genera ~300 esempi di training e ~50 di validazione con frasi realistiche che contengono
documenti d'identità italiani in contesti d'uso comuni (modulistica, corrispondenza, autocertificazioni).

In [4]:
# S3 — Genera dataset Step 1 (documenti d'identità italiani) con split train/val/test
from dataset_builder import build_complete_dataset, label_distribution

bundle = build_complete_dataset(
    n_step1=(1500, 250, 250),   # (train, val, test)
    n_step2=(1000, 200, 200),
    seed=42,
    negative_rate=0.15,          # 15% esempi senza PII (riduce falsi positivi)
)

TRAIN_STEP1 = bundle['step1']['train']
VAL_STEP1   = bundle['step1']['val']
TEST_STEP1  = bundle['step1']['test']

print(f'✅ Step 1 — Train: {len(TRAIN_STEP1)}, Val: {len(VAL_STEP1)}, Test: {len(TEST_STEP1)}')
print('\nDistribuzione label (train):')
for label, cnt in sorted(label_distribution(TRAIN_STEP1).items()):
    print(f'  {label:25s}: {cnt}')

neg = sum(1 for ex in TRAIN_STEP1 if not ex['spans'])
print(f'\nEsempi negativi (senza PII): {neg}/{len(TRAIN_STEP1)} = {100*neg/len(TRAIN_STEP1):.1f}%')


✅ Step 1 — Train: 1500, Val: 250, Test: 250

Distribuzione label (train):
  carta_identita           : 111
  codice_fiscale           : 596
  iban                     : 226
  partita_iva              : 194
  passaporto               : 76
  patente                  : 100
  private_address          : 424
  private_date             : 268
  private_email            : 129
  private_person           : 1200
  private_phone            : 129
  tessera_sanitaria        : 136

Esempi negativi (senza PII): 208/1500 = 13.9%


## 5. Dataset Step 2 — Dominio legale italiano

Genera ~250 esempi con frasi tipiche di atti notarili, contratti, sentenze e procure,
usando il registro linguistico formale del diritto italiano.

In [5]:
# S4 — Dataset Step 2 (dominio legale) — già generato nel bundle
TRAIN_STEP2 = bundle['step2']['train']
VAL_STEP2   = bundle['step2']['val']
TEST_STEP2  = bundle['step2']['test']

print(f'✅ Step 2 — Train: {len(TRAIN_STEP2)}, Val: {len(VAL_STEP2)}, Test: {len(TEST_STEP2)}')
print('\nDistribuzione label (train):')
for label, cnt in sorted(label_distribution(TRAIN_STEP2).items()):
    print(f'  {label:25s}: {cnt}')

neg = sum(1 for ex in TRAIN_STEP2 if not ex['spans'])
print(f'\nEsempi negativi (senza PII): {neg}/{len(TRAIN_STEP2)} = {100*neg/len(TRAIN_STEP2):.1f}%')


✅ Step 2 — Train: 1000, Val: 200, Test: 200

Distribuzione label (train):
  carta_identita           : 28
  codice_fiscale           : 449
  iban                     : 59
  numero_procedimento      : 344
  parte_in_causa           : 542
  partita_iva              : 86
  private_address          : 469
  private_date             : 351
  private_person           : 826
  riferimento_catastale    : 155

Esempi negativi (senza PII): 139/1000 = 13.9%


## 6. Validazione e salvataggio dataset

Verifica entrambi i dataset e li salva su file `.jsonl` pronti per il training.

In [6]:
# S5 — Validazione di tutti e 6 gli split
from dataset_builder import validate_spans

total = 0
for step, splits in [('step1', {'train': TRAIN_STEP1, 'val': VAL_STEP1, 'test': TEST_STEP1}),
                     ('step2', {'train': TRAIN_STEP2, 'val': VAL_STEP2, 'test': TEST_STEP2})]:
    for split_name, data in splits.items():
        errs = validate_spans(data, f'{step}_{split_name}')
        print(f'  {step}_{split_name:6s}: {errs} errori ({len(data)} esempi)')
        total += errs

if total == 0:
    import json
    print('\n✅ Tutti i dataset sono validi!')
    print('\nPreview primo esempio step1_train:')
    print(json.dumps(TRAIN_STEP1[0], indent=2, ensure_ascii=False))
else:
    print(f'\n⚠️  {total} errori totali')


  step1_train : 0 errori (1500 esempi)
  step1_val   : 0 errori (250 esempi)
  step1_test  : 0 errori (250 esempi)
  step2_train : 0 errori (1000 esempi)
  step2_val   : 0 errori (200 esempi)
  step2_test  : 0 errori (200 esempi)

✅ Tutti i dataset sono validi!

Preview primo esempio step1_train:
{
  "text": "Il sig. Davide Ferrara, C.F. FRRDVD61E22H501F, nato il 22/05/1961 a Roma, chiede il rilascio del documento.",
  "spans": {
    "private_person": [
      [
        8,
        22
      ]
    ],
    "codice_fiscale": [
      [
        29,
        45
      ]
    ],
    "private_date": [
      [
        55,
        65
      ]
    ],
    "private_address": [
      [
        68,
        72
      ]
    ]
  }
}


In [7]:
# S5b — Salva tutti e 6 i file .jsonl
from dataset_builder import write_splits_to_disk

paths = write_splits_to_disk(bundle, base_dir='datasets')
print('File scritti:')
for key, (path, n, size) in paths.items():
    print(f'  {path:40s}  {n:>4} esempi  {size/1024:.1f} KB')

import json
print('\nPreview primo esempio step1_train:')
print(json.dumps(TRAIN_STEP1[0], indent=2, ensure_ascii=False))
print('\nPreview primo esempio step2_train:')
print(json.dumps(TRAIN_STEP2[0], indent=2, ensure_ascii=False))


File scritti:
  datasets/step1_train.jsonl                1500 esempi  278.3 KB
  datasets/step1_val.jsonl                   250 esempi  44.0 KB
  datasets/step1_test.jsonl                  250 esempi  45.5 KB
  datasets/step2_train.jsonl                1000 esempi  269.2 KB
  datasets/step2_val.jsonl                   200 esempi  53.1 KB
  datasets/step2_test.jsonl                  200 esempi  52.1 KB

Preview primo esempio step1_train:
{
  "text": "Il sig. Davide Ferrara, C.F. FRRDVD61E22H501F, nato il 22/05/1961 a Roma, chiede il rilascio del documento.",
  "spans": {
    "private_person": [
      [
        8,
        22
      ]
    ],
    "codice_fiscale": [
      [
        29,
        45
      ]
    ],
    "private_date": [
      [
        55,
        65
      ]
    ],
    "private_address": [
      [
        68,
        72
      ]
    ]
  }
}

Preview primo esempio step2_train:
{
  "text": "Atto di pignoramento: si procede al pignoramento dei beni di Federica Santoro, C.F. SNTFRC

## 7. Training Step 1 — Documenti italiani

Parte dal modello base `openai/privacy-filter` e fa finetuning sui documenti italiani.

In [8]:
# S6 — Lancio training Step 1 (auto-scrive label space, non dipende da S1)
import subprocess, os, json
from dataset_builder import LABEL_SPACE

# Scrivi il label space dal modulo — garantisce consistenza e presenza del file
LABEL_SPACE_PATH = 'custom_label_space.json'
with open(LABEL_SPACE_PATH, 'w') as f:
    json.dump(LABEL_SPACE, f, indent=2)
print(f'✅ Label space scritto: {LABEL_SPACE_PATH} ({len(LABEL_SPACE["span_class_names"])} categorie)')

OUTPUT_STEP1 = './checkpoint_step1_italian_docs'
NUM_EPOCHS_STEP1 = 15
BATCH_SIZE_STEP1 = 4   # M4 Pro/Max: prova 8-16

cmd = [
    'opf', 'train', 'datasets/step1_train.jsonl',
    '--validation-dataset', 'datasets/step1_val.jsonl',
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', OUTPUT_STEP1,
    '--epochs', str(NUM_EPOCHS_STEP1),
    '--batch-size', str(BATCH_SIZE_STEP1),
    '--device', DEVICE_CLI,
]

env = os.environ.copy()
if DEVICE_CLI == 'mps':
    env['OPF_MOE_TRITON'] = '0'
    cmd += ['--output-param-dtype', 'bf16']
    print('🍎 Ottimizzazioni M-series attive')

print('🚀 Step 1 — Training su documenti d\'identità italiani...')
print('Comando:', ' '.join(cmd))
print('=' * 60)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print('\n✅ Step 1 completato! Checkpoint:', OUTPUT_STEP1)
else:
    print(f'\n❌ Step 1 fallito (exit code {process.returncode})')


✅ Label space scritto: custom_label_space.json (19 categorie)
🍎 Ottimizzazioni M-series attive
🚀 Step 1 — Training su documenti d'identità italiani...
Comando: opf train datasets/step1_train.jsonl --validation-dataset datasets/step1_val.jsonl --label-space-json custom_label_space.json --output-dir ./checkpoint_step1_italian_docs --epochs 15 --batch-size 4 --device mps --output-param-dtype bf16
info: rebuilt output head for target label space (73 labels; copied_rows=73; exact=33; fallback=40)
training plan: epochs=15 train_examples=1500 train_windows=1500 validation_examples=250 validation_windows=250 progress_interval_s=15.0
Traceback (most recent call last):
  File "/Users/gpuzio/Desktop/CODE/privacy-filter-it/.venv/bin/opf", line 6, in <module>
    sys.exit(main())
             ~~~~^^
  File "/Users/gpuzio/Desktop/CODE/privacy-filter-it/.venv/lib/python3.14/site-packages/opf/__main__.py", line 204, in main
    _run_train_command(subcommand_argv)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^

## 8. Training Step 2 — Dominio legale

Parte dal checkpoint Step 1 già adattato all'italiano e fa finetuning sul dominio legale.

> ⚠️ Eseguire **dopo** la cella S6 e solo se Step 1 ha completato con successo.

In [9]:
# S7 — Lancio training Step 2 (parte da checkpoint Step 1, auto-scrive label space)
import subprocess, os, json
from dataset_builder import LABEL_SPACE

# Garantisce presenza del label space file anche se S1/S6 non sono stati eseguiti
LABEL_SPACE_PATH = 'custom_label_space.json'
with open(LABEL_SPACE_PATH, 'w') as f:
    json.dump(LABEL_SPACE, f, indent=2)

# Verifica che Step 1 sia completato
if not os.path.isdir(OUTPUT_STEP1):
    raise RuntimeError(f'Checkpoint Step 1 non trovato: {OUTPUT_STEP1}. Esegui prima S6.')

OUTPUT_STEP2 = './checkpoint_step2_legal'
NUM_EPOCHS_STEP2 = 12
BATCH_SIZE_STEP2 = 4

cmd = [
    'opf', 'train', 'datasets/step2_train.jsonl',
    '--validation-dataset', 'datasets/step2_val.jsonl',
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', OUTPUT_STEP2,
    '--epochs', str(NUM_EPOCHS_STEP2),
    '--batch-size', str(BATCH_SIZE_STEP2),
    '--device', DEVICE_CLI,
]

# opf train parte da un checkpoint pre-esistente tramite --checkpoint
if os.path.exists(os.path.join(OUTPUT_STEP1, 'model.safetensors')):
    cmd += ['--checkpoint', OUTPUT_STEP1]
    print('ℹ️  Partendo dal checkpoint Step 1:', OUTPUT_STEP1)
else:
    print('ℹ️  Checkpoint Step 1 non trovato, partendo dal modello base openai/privacy-filter')

env = os.environ.copy()
if DEVICE_CLI == 'mps':
    env['OPF_MOE_TRITON'] = '0'
    cmd += ['--output-param-dtype', 'bf16']
    print('🍎 Ottimizzazioni M-series attive')

print('🚀 Step 2 — Training dominio legale...')
print('Comando:', ' '.join(cmd))
print('=' * 60)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print('\n✅ Step 2 completato! Checkpoint finale:', OUTPUT_STEP2)
    FINAL_CHECKPOINT = OUTPUT_STEP2
else:
    print(f'\n❌ Step 2 fallito (exit code {process.returncode})')


RuntimeError: Checkpoint Step 1 non trovato: ./checkpoint_step1_italian_docs. Esegui prima S6.

## 9. Test qualitativo del modello

Prova il modello finale su esempi reali di documenti italiani e testi legali.

In [ ]:
# S8 — Test finale del modello su casi reali
from transformers import pipeline
import os

# DEVICE_HF definito nella cella 2
FINAL_CHECKPOINT = OUTPUT_STEP2 if os.path.isdir(OUTPUT_STEP2) else OUTPUT_STEP1

print(f'Caricamento modello da: {FINAL_CHECKPOINT}')
model_final = pipeline(
    task='token-classification',
    model=FINAL_CHECKPOINT,
    aggregation_strategy='simple',
    device=DEVICE_HF
)
print('✅ Modello caricato\n')

# ── Frasi di test ──
TEST_CASES = [
    # Documenti italiani
    "Il sottoscritto Mario Rossi, codice fiscale RSSMRA80A01H501U, chiede il rilascio del certificato.",
    "Carta d'identità n. AB1234567, intestata a Giulia Bianchi, nata a Milano il 15/03/1990.",
    "Patente di guida: CD456EF78901G — titolare: Luca Ferrari, residente a Roma.",
    "Passaporto n. YK9876543 del sig. Paolo Esposito, valido fino al 20/06/2030.",
    "Partita IVA: 12345678901 — Intestatario: Anna Colombo, via Roma 10, Torino.",
    "Bonifico su IBAN IT60X0542811101000000123456 intestato a Marco Ricci.",
    "Tessera sanitaria n. 8003853450RSSMRA80A01H501U — paziente: Mario Rossi.",
    # Dominio legale
    "Avanti a me, Notaio dott. Carlo Verdi, sono comparsi: il sig. Antonio Russo, C.F. RSSNTN75C15F205K, e la sig.ra Elena Gallo.",
    "Nel procedimento n. 1234/2023 RG pendente avanti al Tribunale di Roma, tra Francesca Mancini (attore) e Roberto Costa (convenuto).",
    "L'immobile censito come foglio 45, mappale 123, sub. 7, sito nel Comune di Napoli, è di proprietà di Giovanni Bruno.",
    "Il mutuatario Stefano Lombardi, C.F. LMBSFN85D20L219Z, si obbliga al pagamento su IBAN IT40V0300203280987654321000.",
]

def redact(text, clf):
    res = sorted(clf(text), key=lambda x: x['start'], reverse=True)
    t = text
    for r in res:
        t = t[:r['start']] + f'[{r["entity_group"]}]' + t[r['end']:]
    return t

print('=' * 70)
for i, text in enumerate(TEST_CASES, 1):
    print(f'[{i:2d}] Input   : {text}')
    print(f'     Redatto : {redact(text, model_final)}')
    print()

## 10. Valutazione quantitativa sul test set held-out

Calcola **precision / recall / F1** per ogni categoria sul test set held-out.
Il test set non è mai stato visto dal modello durante il training, quindi queste metriche sono una stima onesta della qualità del fine-tuning.

- **Precision**: delle entità predette, quante sono corrette?
- **Recall**: delle entità reali, quante ne trova?
- **F1**: media armonica di precision e recall (metrica principale).

Soglia di "successo" indicativa: F1 **> 0.85** sulle categorie principali.

In [ ]:
# S8b — Metriche su test set held-out (TEST_STEP1 + TEST_STEP2)
from collections import defaultdict

def compute_span_metrics(classifier, test_data, match='exact'):
    """Calcola TP/FP/FN per label su test_data.

    match='exact': span perfettamente uguali (label, start, end)
    match='overlap': basta sovrapposizione + label uguale
    """
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)

    for ex in test_data:
        text = ex['text']
        gold = set()
        for label, spans in ex.get('spans', {}).items():
            for s, e in spans:
                gold.add((label, s, e))

        preds = set()
        for r in classifier(text):
            preds.add((r['entity_group'], int(r['start']), int(r['end'])))

        if match == 'exact':
            matched_gold = preds & gold
            for p in preds:
                (tp if p in gold else fp)[p[0]] += 1
            for g in gold:
                if g not in preds:
                    fn[g[0]] += 1
        else:  # overlap
            used_gold = set()
            for p_lab, p_s, p_e in preds:
                match_found = False
                for g in gold - used_gold:
                    g_lab, g_s, g_e = g
                    if g_lab == p_lab and not (p_e <= g_s or g_e <= p_s):
                        tp[p_lab] += 1
                        used_gold.add(g)
                        match_found = True
                        break
                if not match_found:
                    fp[p_lab] += 1
            for g_lab, g_s, g_e in gold - used_gold:
                fn[g_lab] += 1

    return tp, fp, fn


def print_metrics_table(tp, fp, fn, title=''):
    labels = sorted(set(tp) | set(fp) | set(fn))
    tot_tp = sum(tp.values()); tot_fp = sum(fp.values()); tot_fn = sum(fn.values())

    print(f'\n{title}')
    print(f'{"label":25s} {"TP":>5s} {"FP":>5s} {"FN":>5s} {"P":>7s} {"R":>7s} {"F1":>7s}')
    print('─' * 70)
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] > 0 else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] > 0 else 0.0
        f1 = 2*p*r/(p+r) if p+r > 0 else 0.0
        marker = ' ✅' if f1 >= 0.85 else (' ⚠️ ' if f1 >= 0.7 else ' ❌')
        print(f'{l:25s} {tp[l]:>5d} {fp[l]:>5d} {fn[l]:>5d} {p:>7.3f} {r:>7.3f} {f1:>7.3f}{marker}')

    mp = tot_tp / (tot_tp + tot_fp) if tot_tp + tot_fp > 0 else 0.0
    mr = tot_tp / (tot_tp + tot_fn) if tot_tp + tot_fn > 0 else 0.0
    mf1 = 2*mp*mr/(mp+mr) if mp+mr > 0 else 0.0
    print('─' * 70)
    print(f'{"MICRO AVG":25s} {tot_tp:>5d} {tot_fp:>5d} {tot_fn:>5d} {mp:>7.3f} {mr:>7.3f} {mf1:>7.3f}')
    return mf1


# Assicurati che model_final sia caricato (dalla cella S8)
print('Valutazione in corso su test set held-out...')
print('Match: EXACT (span perfettamente coincidenti, label compresa)\n')

tp1, fp1, fn1 = compute_span_metrics(model_final, TEST_STEP1, match='exact')
f1_s1 = print_metrics_table(tp1, fp1, fn1, title=f'═══ TEST STEP 1 — Documenti italiani ({len(TEST_STEP1)} esempi) ═══')

tp2, fp2, fn2 = compute_span_metrics(model_final, TEST_STEP2, match='exact')
f1_s2 = print_metrics_table(tp2, fp2, fn2, title=f'\n═══ TEST STEP 2 — Dominio legale ({len(TEST_STEP2)} esempi) ═══')

print(f'\n{"═"*70}')
print(f'F1 micro Step 1: {f1_s1:.3f}')
print(f'F1 micro Step 2: {f1_s2:.3f}')
print(f'F1 micro medio:  {(f1_s1+f1_s2)/2:.3f}')
print(f'{"═"*70}')

# Anche con match overlap (più permissivo)
print('\n\nValutazione con MATCH OVERLAP (span che si sovrappongono):\n')
tp1o, fp1o, fn1o = compute_span_metrics(model_final, TEST_STEP1, match='overlap')
print_metrics_table(tp1o, fp1o, fn1o, title='═══ TEST STEP 1 (overlap) ═══')
tp2o, fp2o, fn2o = compute_span_metrics(model_final, TEST_STEP2, match='overlap')
print_metrics_table(tp2o, fp2o, fn2o, title='\n═══ TEST STEP 2 (overlap) ═══')


## 11. Salvataggio checkpoint finale

In [ ]:
# S9 — Salva entrambi i checkpoint (Drive su Colab, cartella locale altrove)
import shutil, os

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DEST = '/content/drive/MyDrive/privacy_filter'
else:
    BASE_DEST = os.path.expanduser('~/Documents/privacy_filter_checkpoints')

os.makedirs(BASE_DEST, exist_ok=True)

for src, name in [(OUTPUT_STEP1, 'checkpoint_step1_italian_docs'),
                   (OUTPUT_STEP2, 'checkpoint_step2_legal'),
                   ('datasets', 'datasets')]:
    dest = os.path.join(BASE_DEST, name)
    if os.path.exists(src):
        if os.path.isdir(dest):
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        print(f'✅ Salvato: {dest}')
    else:
        print(f'⚠️  Non trovato (saltato): {src}')

print('\nTutto salvato in:', BASE_DEST)
